# refusal_geometry - Kaggle bootstrap (Gate M1)

Runs the M1 validation on **Kaggle dual-T4**: clone repo -> install env -> download
benchmarks -> verify the zoo loads & generates deterministically -> first reconciled
refusal numbers. The 70B int8 point is **not** here (it belongs to the rented A100 window).

**Before running:**
1. Settings -> Accelerator -> **GPU T4 x2**.
2. Add-ons -> Secrets -> add:
   - `GITHUB_TOKEN` - a GitHub PAT with read access to the private repo.
   - `HF_TOKEN` - a Hugging Face token that has accepted the Llama-3 / Mistral licenses
     (gated). Without it those models are skipped.
3. Settings -> Persistence -> **Files** (so `/kaggle/working` survives session restarts;
   the run_id-keyed resume then skips finished units).

In [ ]:
# 1. Clone (or update) the private repo using the GITHUB_TOKEN secret.
import os
from kaggle_secrets import UserSecretsClient

sec = UserSecretsClient()
gh = sec.get_secret("GITHUB_TOKEN")
try:
    os.environ["HF_TOKEN"] = sec.get_secret("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
except Exception:
    print("No HF_TOKEN secret -> gated models (Llama-3, Mistral) will be skipped.")

REPO = "/kaggle/working/refusal_geometry"
if not os.path.exists(REPO):
    !git clone https://{gh}@github.com/SLIMIHamda/refusal_geometry.git {REPO}
else:
    !cd {REPO} && git pull --ff-only
%cd {REPO}

## 2. Environment (zoo env: bumped for Qwen-2.5)

Kaggle ships torch (cu121) preinstalled. We bump `transformers>=4.43` so Qwen-2.5 loads -
this is the deliberate two-env split flagged in `requirements.txt`; the thesis pin
(transformers 4.40.0) only covers Llama-3 / Mistral.

In [ ]:
# 2. Install the GPU stack + harness deps. (torch stays as Kaggle's preinstalled cu121 build.)
!pip install -q "transformers==4.44.2" "accelerate==0.33.0" "bitsandbytes==0.43.1"     "datasets==2.20.0" pyyaml pandas pyarrow scipy scikit-learn

# Determinism flags (mirror asw.repro.set_seed for the shell-invoked steps).
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "0"

import torch
print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 2b. Confirm the GPU-free spine is green on this host before touching models.
!python -m pytest -q

## 3. Download benchmarks -> `data/*.jsonl`

Best-effort HF mirror ids live in `asw/data/download.py::SPECS`; `[fail]` lines below mean an
id/field moved - fix the SPEC and re-run just that one. `--limit` caps each set for the M1 pass.

In [ ]:
# 3. Populate data/. Bump/remove --limit for the full sets later.
!python scripts/download_benchmarks.py --all --limit 300
!ls -la data/*.jsonl

## 4. Gate M1 - zoo loads & generates deterministically

`models-verify` loads each model and checks T=0 (greedy) generation is identical across two
runs. 70B is skipped (paid window). 14B fits bf16 across 2x T4 (~28 GB). Gated models without
a valid `HF_TOKEN` will error - that's expected, note which ones and proceed.

In [ ]:
# 4. Verify every <=14B config. Records nothing to the DB; this is a load/determinism smoke.
import glob, subprocess

configs = [c for c in sorted(glob.glob("configs/models/*.yaml")) if "70b" not in c]
results = {}
for cfg in configs:
    print("=" * 72, "
", cfg)
    r = subprocess.run(["python", "-m", "asw.harness.cli", "models-verify", "--config", cfg])
    results[cfg] = r.returncode
print("
=== verify summary (0 = deterministic load OK) ===")
for c, rc in results.items():
    print(f"  {rc}  {c}")

## 5. First reconciled numbers - sample `asw eval`

Writes a run row to `results/runs.sqlite` + per-prompt parquet. This is the M1 'one config'
reconciliation entry point: pick the AdvBench/HarmBench config here and every later number
reuses it. Re-running skips finished (model, config, seed) units via the deterministic run_id.

In [ ]:
# 5. Sample eval on the reference model. Add --hf-judge to also run scorer #2 (RoBERTa).
!python -m asw.harness.cli eval     --config configs/models/dolphin-llama3-8b.yaml     --benchmark harmbench --limit 100 --seeds 0 1 2

In [ ]:
# 5b. Inspect the run manifest (Tab A) - every paper number traces back to a row here.
import sqlite3, pandas as pd

con = sqlite3.connect("results/runs.sqlite")
df = pd.read_sql(
    "SELECT experiment, model_id, seed, config_hash, status, "
    "wall_clock_s, gpu_type, metrics_json FROM runs ORDER BY started_at", con)
pd.set_option("display.max_colwidth", 60)
print(df.to_string())

## 6. Persist & resume

`results/` and `data/` live under `/kaggle/working`, so committing the notebook (Save Version)
keeps them. On the next session, re-run cells 1-4 then **re-run cell 5** - completed units are
skipped, unfinished prompts resume from the parquet. Nothing recomputes.

**Next (Step 5, gate M2):** behavioral-contrast extraction -> the anti-alignment map.